In [0]:
import requests
import json
import boto3
from pyspark.sql import functions as F
from pyspark.sql.types import (
    ArrayType, StructType, StructField,
    StringType, DoubleType
)

In [0]:
def pn_api_extraction(api_url,api_param,headers,offset,limit):

    all_barca_players = []    
    
    while True:
        api_param['limit'] = limit
        api_param['offset'] = offset

        res = requests.get(api_url, params=api_param, headers=headers)
        data = res.json()
       
        if top_lvl_filter is not None:
            data = data.get(top_lvl_filter, [])

        # Stop when no more data
        if not data and isinstance(data, dict):
            break
        if len(data) == 0 and isinstance(data, list):
            break
        for p in data:
            all_barca_players.append(p)

        offset += limit 
    json_data = json.dumps(all_barca_players, indent=4, ensure_ascii=False)
    
    return json_data

def api_extraction(api_url,api_param):

    all_barca_players = []

    response = requests.get(api_url, params=api_param)
    json_data = response.json()

    if top_lvl_filter is not None:
        json_data = json_data.get(top_lvl_filter, [])
    
    json_data = json.dumps(json_data, indent=4, ensure_ascii=False)
    
    return json_data

In [0]:
def upload_s3_vol(json_data,bucket_name,s3_key,volume_path):
    s3_error_message = {}
    dbk_error_message = {}

    try:
        # Create S3 client
        s3 = boto3.client(
            "s3",
            aws_access_key_id=access_key,
            aws_secret_access_key=secret_key,
            region_name="us-east-1"
        )

        # Upload to S3
        s3.put_object(
            Bucket=bucket_name,
            Key=s3_key,
            Body=json_data,
            ContentType="application/json"
        )
        print(f"File uploaded to S3 the path:{s3_key}")
    except Exception as e:
        s3_error_message[file_name] = f"Error uploading file to S3: {str(e)}"
       
    try:
        response = requests.put(
            f"{databricks_host}/api/2.0/fs/files{volume_path}",
            headers={
                "Authorization": f"Bearer {databricks_token}",
                "Content-Type": "application/octet-stream"
            },
            data=json_data.encode("utf-8")
        )

        print(f"File uploaded to volume {volume_path}")
    except Exception as e:
        dbk_error_message[file_name] = f"Error uploading file to volume: {str(e)}"

In [0]:
def raw_file_to_table(data,file_name):
    
    selected_data = []
    col_len = len(list(data[0].keys()))
    ref_col = list(data[0].keys())

    for row in data:
        record = {}
        columns = list(row.keys())
        if(len(columns) > col_len):
            ref_col = columns

        for col in columns:
            value = row.get(col)

            if isinstance(value, (dict, list)):
                value = json.dumps(value, ensure_ascii=False)

            record[col] = value

        selected_data.append(record)

    df_file = spark.createDataFrame(selected_data).select(ref_col)
    
    (
        df_file.write.mode("overwrite")
        .option("overwriteSchema", "true")
        .saveAsTable(f"workspace.bk_raw.{file_name}")
    )

In [0]:
def expand_json_columns(raw_df):
    # Maintain one DataFrame throughout JSON column processing
    working_df = raw_df

    # Store all dynamically extracted column names
    extracted_columns = []

    json_columns_to_process = [
        c.strip() for c in json_col.split(",") if c.strip()
    ]

    for jc in json_columns_to_process:

        # Get a non-null JSON sample
        json_sample_row = (
            working_df
            .filter(col(jc).isNotNull())
            .select(jc)
            .first()
        )

        if json_sample_row is None:
            print(f"No JSON data found in {jc}. Skipping.")
            continue

        json_sample = json_sample_row[jc]

        # Dynamically infer schema
        schema_ddl = (
            spark.range(1)
            .select(schema_of_json(lit(json_sample)))
            .first()[0]
        )

        json_schema = StructType.fromDDL(schema_ddl)

        # Convert JSON string into struct
        struct_col = f"{jc}_struct"

        working_df = working_df.withColumn(
            struct_col,
            from_json(col(jc), json_schema)
        )

        # Extract JSON fields dynamically
        json_field_columns = [
            col(f"{struct_col}.{field.name}").alias(f"{jc}_{field.name}")
            for field in json_schema.fields
        ]

        extracted_columns.extend(
            [f"{jc}_{field.name}" for field in json_schema.fields]
        )

        # Update working_df instead of recreating it from raw_df
        working_df = (
            working_df
            .select("*", *json_field_columns)
            .drop(struct_col)
        )


    return working_df

In [0]:
def expand_list_json_columns(df, list_json_col):

    json_columns = [
        c.strip()
        for c in list_json_col.split(",")
        if c.strip()
    ]

    df = df.withColumn(
        "_row_id",
        F.monotonically_increasing_id()
    )

    for json_column in json_columns:

        column_type = df.schema[json_column].dataType

        # Parse JSON string and infer its schema dynamically
        if isinstance(column_type, StringType):

            json_schema = F.schema_of_json(
                df.select(json_column)
                  .filter(F.col(json_column).isNotNull())
                  .first()[0]
            )

            df = df.withColumn(
                json_column,
                F.from_json(F.col(json_column), json_schema)
            )

        elif not isinstance(column_type, ArrayType):
            raise TypeError(
                f"{json_column} must be STRING or ARRAY"
            )

        # Identify fields dynamically from the array element schema
        element_fields = df.schema[json_column].dataType.elementType.fieldNames()

        key_field = element_fields[0]
        value_field = element_fields[1]

        exploded_df = df.select(
            "_row_id",
            F.explode_outer(F.col(json_column)).alias("exploded")
        )

        flattened_df = exploded_df.filter(
            F.col("exploded").isNotNull()
        ).select(
            "_row_id",
            F.concat(
                F.lit(json_column + "_"),
                F.col(f"exploded.{key_field}").cast("string")
            ).alias("column_name"),
            F.col(f"exploded.{value_field}").alias("column_value")
        )

        pivot_df = flattened_df.groupBy(
            "_row_id"
        ).pivot("column_name").agg(
            F.first("column_value")
        )

        df = df.join(pivot_df, "_row_id", "left")

        df = df.drop(json_column)

    return df.drop("_row_id")